In [ ]:
!pip install transformers torch torchvision -q

In [ ]:
import os
import zipfile
import numpy as np
import torch

from transformers import AutoImageProcessor, AutoModel
from PIL import Image
from tqdm import tqdm

In [ ]:
zip_path = "/content/image_dataset_clean.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

print("Dataset extracted ✅")

Dataset extracted ✅


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'image_dataset_clean.zip', 'diagrams', 'data_visualization', 'tables', 'sample_data']


In [ ]:
dataset_path = "/content"

In [ ]:
processor = AutoImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

vit_model = AutoModel.from_pretrained(
    "google/vit-base-patch16-224"
)

vit_model.eval()

print("ViT model loaded successfully ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT model loaded successfully ✅


In [ ]:
image_paths = []
labels = []

for label in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, label)

    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            image_paths.append(os.path.join(class_path, img_name))
            labels.append(label)

print("Total images:", len(image_paths))

Total images: 8083


In [ ]:
embeddings = []

for img_path in tqdm(image_paths):

    try:
        image = Image.open(img_path).convert("RGB")

        inputs = processor(images=image, return_tensors="pt")

        with torch.no_grad():
            outputs = vit_model(**inputs)

        cls_emb = outputs.last_hidden_state[:, 0, :]

        embeddings.append(cls_emb.numpy().flatten())

    except:
        continue

100%|██████████| 8083/8083 [1:32:06<00:00,  1.46it/s]


In [ ]:
vit_embeddings = np.array(embeddings)

print("Shape:", vit_embeddings.shape)

Shape: (8067, 768)


In [ ]:
np.save("vit_image_embeddings.npy", vit_embeddings)

print("Saved embeddings ✅")

Saved embeddings ✅


In [ ]:
np.save("vit_labels.npy", labels)

print("Saved labels ✅")

Saved labels ✅


## 📊 Model Comparison: CNN vs EfficientNetB0 vs Vision Transformer (ViT)

This section compares three different deep learning approaches used for image feature extraction and classification:

- CNN (Custom baseline model)
- EfficientNetB0 (Transfer Learning model)
- Vision Transformer (ViT)

---

### 📈 Performance Comparison

| Metric | CNN Baseline | EfficientNetB0 | Vision Transformer (ViT) | Insight |
|---|---|---|---|---|
| Model Type | Custom CNN | Pretrained CNN (Transfer Learning) | Transformer-based architecture | ViT uses attention instead of convolution |
| Training Strategy | From scratch | Fine-tuned pretrained weights | Pretrained feature extractor (no full training shown) | Transfer learning improves performance |
| Accuracy | 86.8% | ~99.5% – 99.7% | Not explicitly trained classifier (feature extraction only) | EfficientNet leads in classification |
| Validation Accuracy | 86.8% | ~93% – 97% | N/A (used for embeddings) | CNN underperforms vs transfer learning |
| Loss | 0.6366 | 0.16 – 0.20 | Not computed | EfficientNet shows better optimization |
| Feature Representation | Local spatial features | Strong hierarchical features | Global contextual features (CLS token) | ViT captures global relationships |
| Embedding Dimension | Low-level feature maps | High-level feature vectors | 768-dimensional embeddings | ViT produces rich semantic vectors |
| Convergence Speed | Medium | Fast | Immediate inference (pretrained) | Transformers are fast for inference |
| Overfitting Risk | Medium | Low | Low (feature extractor) | Pretrained models generalize better |

---

### 🧠 Key Insights

#### 1. CNN (Baseline Model)
- Works as a simple baseline architecture
- Learns features from scratch
- Performs well but limited in complex pattern understanding
- More prone to overfitting compared to pretrained models

---

#### 2. EfficientNetB0
- Best performing model in classification task
- Uses transfer learning → leverages ImageNet knowledge
- Strong generalization on validation data
- More stable and accurate than CNN

---

#### 3. Vision Transformer (ViT)
- Uses attention mechanism instead of convolution
- Produces **high-dimensional embeddings (768 features)**
- Best suited for:
  - similarity search
  - semantic understanding
  - clustering tasks
- Not directly compared in accuracy because it is used as a feature extractor

---

### 🏆 Final Conclusion

- **Best classification model:** EfficientNetB0  
- **Best feature representation model:** Vision Transformer (ViT)  
- **Baseline performance:** CNN

---

### 🔥 Final Insight

EfficientNet is the strongest model for supervised classification, while ViT provides the richest semantic embeddings suitable for advanced tasks like similarity matching, retrieval systems, and clustering.

A hybrid system combining **EfficientNet for classification + ViT for embedding similarity** would achieve the best overall performance.